# Perturb-FISH — data preparation & LUNA training launch

## 1. Generate the data splits & LUNA-formatted CSVs

Run `data-sampler.py` to create:
1. **Raw splits** in `data_splits/` (intermediate, headerless)
2. **Merged splits** in `data_splits/merged_splits/` (with headers + `cell_section` + `cell_class`)
3. **LUNA-ready files** in `data_splits/luna/` (`perturb_fish_train.csv` and `perturb_fish_test.csv`)

In [ ]:
%run data-sampler.py

## 2. Quick sanity check

Verify that the LUNA files have the expected columns and shape.

In [7]:
import pandas as pd

luna_dir = "data_splits/luna"
train = pd.read_csv(f"{luna_dir}/perturb_fish_train.csv", index_col=0)
test  = pd.read_csv(f"{luna_dir}/perturb_fish_test.csv",  index_col=0)

required = ["coord_X", "coord_Y", "cell_section", "cell_class"]
for name, df in [("Train", train), ("Test", test)]:
    assert all(c in df.columns for c in required), f"{name} missing required cols!"
    print(f"{name}: {df.shape[0]:,} cells × {df.shape[1]} cols")
    print(f"  First 5 cols : {list(df.columns[:5])}")
    print(f"  Last 5 cols  : {list(df.columns[-5:])}")
    print(f"  coord_X range: [{df['coord_X'].min():.1f}, {df['coord_X'].max():.1f}]")
    print()

Train: 93,529 cells × 631 cols
  First 5 cols : ['PDK4', 'CCL26', 'CX3CL1', 'PGLYRP1', 'CD4']
  Last 5 cols  : ['gRNA_76', 'coord_X', 'coord_Y', 'cell_section', 'cell_class']
  coord_X range: [36.9, 91986.5]

Test: 93,686 cells × 631 cols
  First 5 cols : ['PDK4', 'CCL26', 'CX3CL1', 'PGLYRP1', 'CD4']
  Last 5 cols  : ['gRNA_76', 'coord_X', 'coord_Y', 'cell_section', 'cell_class']
  coord_X range: [0.0, 0.0]



## 3. Launch LUNA training

The command below runs LUNA's `main.py` using the Hydra config under `configs/`.
The experiment config `configs/experiment/perturb-FISH.yaml` is already set for a **CPU smoke-test** (3 epochs, batch size 1, wandb disabled, graphs downsampled to 5 000 cells).

For a real GPU run later, edit the yaml:
- `train.n_epochs: 1000`
- `train.batch_size: 6`
- `distribute.gpus_per_node: [0]`
- `dataset.maximum_graph_size.train: null`
- `general.wandb: 'online'`

In [ ]:
import subprocess, sys, os

luna_dir = os.path.abspath("../LUNA")
main_py  = os.path.join(luna_dir, "main.py")
configs  = os.path.abspath("configs")

cmd = [
    sys.executable, main_py,
    f"--config-path={configs}",
    "general.wandb=disabled",
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=luna_dir)
print(f"\nProcess exited with code {result.returncode}")

Running: c:\Users\abels\miniconda3\envs\LUNA\python.exe c:\Users\abels\Documents\Ricci\code\LUNA\main.py --config-path=c:\Users\abels\Documents\Ricci\code\LUNA_perturb\configs general.wandb=disabled

Process exited with code 1
